# 30 - Matched-budget ladder (Phase 2 of the crossover-prediction study)

Phase 1 (notebook 29) showed the naive small-ablation procedure fails badly, but its tiny rungs
(12.3M tokens) and the 29.4M anchor (24.6M tokens) differed in token budget. This notebook removes
that confound: **every rung trains on exactly the anchor's protocol** - 6000 iters x batch 16 x
block 256 = 24.6M tokens, WikiText-103 BPE.

Rungs: 5.25M (256d/4L/4h), 10.0M (320d/6L/8h), 15.5M (384d/7L/8h), 20.5M (448d/7L/8h)
Variants: combo / combo_conv3 / combo_diff. Seeds 0, 1.
One cell per (size, variant): powercut/timeout-safe - per-config CSV skip + atomic checkpoint
every 1000 iters; re-running the notebook resumes exactly where it stopped.


In [1]:
%matplotlib inline
import os, math, time, csv, json, gc
from contextlib import nullcontext
import numpy as np
import torch
from model import Config, GPT, count_params

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
D = "../dataset"
train_data = torch.from_numpy(np.load(D + "/wt103_bpe_train.npy"))
val_data = torch.from_numpy(np.load(D + "/wt103_bpe_val.npy"))
vocab = json.load(open(D + "/wt103_bpe_meta.json"))["vocab"]
print(f"vocab={vocab}  train={len(train_data):,}  val={len(val_data):,}")

BLOCK, BATCH = 256, 16
CKPT_DIR = "../results/ladder_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

COMBO = {"pos_encoding": "rope", "activation": "swiglu", "norm_placement": "pre", "norm_type": "rmsnorm"}
SIZES = {"5M": (256, 4, 4), "10M": (320, 6, 8), "15M": (384, 7, 8), "20M": (448, 7, 8)}
VARIANTS = {"combo": COMBO, "combo_conv3": {**COMBO, "conv_kernel": 3}, "combo_diff": {**COMBO, "attention": "diff"}}
SEEDS = [0, 1]
RUNS_CSV = "../results/ladder6000_runs.csv"
if not os.path.exists(RUNS_CSV):
    with open(RUNS_CSV, "w", newline="") as f:
        csv.writer(f).writerow(["size", "variant", "seed", "params_M", "val_ppl"])

def get_batch(split, gen):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK, (BATCH,), generator=gen)
    x = torch.stack([d[i:i + BLOCK] for i in ix]); y = torch.stack([d[i + 1:i + 1 + BLOCK] for i in ix])
    return x.to(device), y.to(device)
def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

def train_one(name, size_cfg, over, seed=0, max_iters=6000, eval_iters=200,
              lr=3e-4, warmup=200, min_lr=3e-5, ckpt_every=1000):
    ne, nl, nh = size_cfg
    ckpt_path = f"{CKPT_DIR}/{name}_s{seed}.pt"
    torch.manual_seed(seed)
    cfg = Config(vocab_size=vocab, block_size=BLOCK, n_layer=nl, n_head=nh, n_embd=ne,
                 dropout=0.1, **over)
    model = GPT(cfg).to(device)
    pm = count_params(model) / 1e6
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.1)
    tg = torch.Generator().manual_seed(7777 + seed)
    start_it = 0
    if os.path.exists(ckpt_path):
        # load to CPU: Generator.set_state() requires a CPU ByteTensor; model/opt states are
        # cast onto the cuda params automatically by their load_state_dict calls
        ck = torch.load(ckpt_path, map_location="cpu")
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        tg.set_state(ck["tg"].cpu()); start_it = ck["iter"]
        print(f"    [{name} s{seed}] RESUMED from {start_it}", flush=True)
    def get_lr(it):
        if it < warmup: return lr * (it + 1) / warmup
        r = (it - warmup) / max(1, max_iters - warmup)
        return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * r))
    def save_ckpt(it):
        tmp = ckpt_path + ".tmp"
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "iter": it, "tg": tg.get_state()}, tmp)
        os.replace(tmp, ckpt_path)
    @torch.no_grad()
    def ev():
        model.eval(); g = torch.Generator().manual_seed(4242); tot = 0.0
        for _ in range(eval_iters):
            x, y = get_batch("val", g)
            with autocast_ctx():
                _, loss = model(x, y)
            tot += loss.item()
        model.train(); return tot / eval_iters
    for it in range(start_it, max_iters):
        for grp in opt.param_groups: grp["lr"] = get_lr(it)
        x, y = get_batch("train", tg)
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if (it + 1) % ckpt_every == 0: save_ckpt(it + 1)
    ppl = math.exp(ev())
    if os.path.exists(ckpt_path): os.remove(ckpt_path)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return ppl, pm

def sweep(sname, vname):
    """Run all seeds for one (size, variant); skip anything already in the CSV."""
    done = set()
    with open(RUNS_CSV) as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if len(row) == 5: done.add((row[0], row[1], row[2]))
    t0 = time.time()
    for s in SEEDS:
        if (sname, vname, str(s)) in done:
            print(f"  {sname} {vname} s{s}: done, skip", flush=True); continue
        p, pm = train_one(f"{sname}_{vname}", SIZES[sname], VARIANTS[vname], seed=s)
        with open(RUNS_CSV, "a", newline="") as f:
            csv.writer(f).writerow([sname, vname, s, f"{pm:.3f}", f"{p:.4f}"])
        print(f"  {sname:4s} {vname:12s} seed {s}: ppl {p:.3f} ({pm:.2f}M) | {(time.time()-t0)/60:.1f} min", flush=True)
    print(f"pair done in {(time.time()-t0)/60:.1f} min", flush=True)

print("ready")


device: cuda
vocab=8192  train=14,923,628  val=281,169
ready


In [2]:
sweep("5M", "combo")

  5M combo s0: done, skip


  5M combo s1: done, skip


pair done in 0.0 min


In [3]:
sweep("5M", "combo_conv3")

  5M combo_conv3 s0: done, skip


  5M combo_conv3 s1: done, skip


pair done in 0.0 min


In [4]:
sweep("5M", "combo_diff")

  5M combo_diff s0: done, skip


  5M combo_diff s1: done, skip


pair done in 0.0 min


In [5]:
sweep("10M", "combo")

  10M combo s0: done, skip


  10M combo s1: done, skip


pair done in 0.0 min


In [6]:
sweep("10M", "combo_conv3")

  10M combo_conv3 s0: done, skip


  10M combo_conv3 s1: done, skip


pair done in 0.0 min


In [7]:
sweep("10M", "combo_diff")

  10M combo_diff s0: done, skip


  10M combo_diff s1: done, skip


pair done in 0.0 min


In [8]:
sweep("15M", "combo")

  15M combo s0: done, skip


  15M combo s1: done, skip


pair done in 0.0 min


In [9]:
sweep("15M", "combo_conv3")

  15M combo_conv3 s0: done, skip


  15M combo_conv3 s1: done, skip


pair done in 0.0 min


In [10]:
sweep("15M", "combo_diff")

    [15M_combo_diff s0] RESUMED from 1000


  15M  combo_diff   seed 0: ppl 70.370 (15.57M) | 7.8 min


  15M  combo_diff   seed 1: ppl 71.101 (15.57M) | 17.1 min


pair done in 17.1 min


In [11]:
sweep("20M", "combo")

  20M  combo        seed 0: ppl 59.969 (20.56M) | 7.2 min


  20M  combo        seed 1: ppl 60.471 (20.56M) | 14.3 min


pair done in 14.3 min


In [12]:
sweep("20M", "combo_conv3")

  20M  combo_conv3  seed 0: ppl 54.475 (20.58M) | 8.0 min


  20M  combo_conv3  seed 1: ppl 54.135 (20.58M) | 16.1 min


pair done in 16.1 min


In [13]:
sweep("20M", "combo_diff")

  20M  combo_diff   seed 0: ppl 62.551 (20.56M) | 10.2 min


  20M  combo_diff   seed 1: ppl 65.393 (20.56M) | 20.5 min


pair done in 20.5 min


In [14]:
# === The clean prediction test: fit MATCHED-BUDGET rungs -> predict 29.4M -> uncover truth ===
lad = {}
pmd = {}
with open("../results/ladder6000_runs.csv") as f:
    r = csv.reader(f); next(r, None)
    for sz, v, s, pm, pp in r:
        lad[(sz, v, s)] = float(pp); pmd[sz] = float(pm)
def load30(fn):
    d = {}
    with open(fn) as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if len(row) == 3: d[row[0]] = float(row[2])
    return d
s0 = load30("../results/scale_confirm_runs.csv"); s1 = load30("../results/scale_confirm_s1_runs.csv")

X30 = math.log10(29.407e6)
order = ["5M", "10M", "15M", "20M"]
print("=== matched-budget ladder (24.6M tokens / 6000 steps / batch 16 everywhere) ===")
print("combo ppl per size: " + "  ".join(
    f"{sz}:{np.mean([lad[(sz,'combo',s)] for s in ('0','1')]):.2f}" for sz in order if (sz,"combo","0") in lad))
for variant, label in [("combo_conv3", "depthwise conv"), ("combo_diff", "differential attn")]:
    pts = []   # (size, log10N, delta%) - seed-paired points
    for sz in order:
        for s in ("0", "1"):
            if (sz, variant, s) in lad and (sz, "combo", s) in lad:
                pts.append((sz, math.log10(pmd[sz] * 1e6),
                            100.0 * (lad[(sz, variant, s)] - lad[(sz, "combo", s)]) / lad[(sz, "combo", s)]))
    per_size = {sz: np.mean([y for z, x, y in pts if z == sz]) for sz in order if any(z == sz for z, x, y in pts)}
    print(f"\n  {label}: per-size Delta% = " + "  ".join(f"{sz}:{d:+.2f}" for sz, d in per_size.items()))
    act = np.mean([100.0*((s0 if i==0 else s1)[variant]-(s0 if i==0 else s1)["combo"])/(s0 if i==0 else s1)["combo"] for i in (0,1)])
    # prediction ablation: which subset of rungs + which fit shape would have called 29.4M correctly?
    subsets = [(["5M","10M"], 1), (["5M","10M","15M"], 1), (["5M","10M","15M"], 2),
               (["5M","10M","15M","20M"], 1), (["5M","10M","15M","20M"], 2)]
    for subset, deg in subsets:
        sx = [x for z, x, y in pts if z in subset]
        sy = [y for z, x, y in pts if z in subset]
        if len(set(sx)) > deg:
            cf = np.polyfit(sx, sy, deg)
            pred = np.polyval(cf, X30)
            nm = f"{'+'.join(subset)} deg{deg}"
            print(f"    fit [{nm:26s}] -> 29.4M: {pred:+7.2f}%  (actual {act:+.2f}%)  err {pred-act:+6.2f}pp"
                  + ("  sign OK" if (pred < 0) == (act < 0) else "  SIGN WRONG"))


=== matched-budget ladder (24.6M tokens / 6000 steps / batch 16 everywhere) ===
combo ppl per size: 5M:82.82  10M:67.93  15M:62.99  20M:60.22

  depthwise conv: per-size Delta% = 5M:+8.12  10M:+5.13  15M:-3.36  20M:-9.82
    fit [5M+10M deg1               ] -> 29.4M:   +0.14%  (actual -10.23%)  err +10.38pp  SIGN WRONG
    fit [5M+10M+15M deg1           ] -> 29.4M:   -8.30%  (actual -10.23%)  err  +1.93pp  sign OK
    fit [5M+10M+15M deg2           ] -> 29.4M:  -24.83%  (actual -10.23%)  err -14.60pp  sign OK
    fit [5M+10M+15M+20M deg1       ] -> 29.4M:  -12.30%  (actual -10.23%)  err  -2.07pp  sign OK
    fit [5M+10M+15M+20M deg2       ] -> 29.4M:  -21.82%  (actual -10.23%)  err -11.58pp  sign OK

  differential attn: per-size Delta% = 5M:+15.72  10M:+21.82  15M:+12.33  20M:+6.22
    fit [5M+10M deg1               ] -> 29.4M:  +32.00%  (actual +3.95%)  err +28.05pp  sign OK
    fit [5M+10M+15M deg1           ] -> 29.4M:  +14.12%  (actual +3.95%)  err +10.16pp  sign OK
    fit [5M+10

In [15]:
# === PRE-REGISTERED predictions for ~90M (tested prospectively in Phase 3) ===
X90 = math.log10(90e6)
print("=== commit these numbers BEFORE any 90M run exists ===")
for variant, label in [("combo_conv3", "depthwise conv"), ("combo_diff", "differential attn")]:
    xs, ys = [], []
    for sz in order:
        for s in ("0", "1"):
            if (sz, variant, s) in lad and (sz, "combo", s) in lad:
                xs.append(math.log10(pmd[sz] * 1e6))
                ys.append(100.0 * (lad[(sz, variant, s)] - lad[(sz, "combo", s)]) / lad[(sz, "combo", s)])
    for i, dd in ((0, s0), (1, s1)):
        xs.append(X30); ys.append(100.0*(dd[variant]-dd["combo"])/dd["combo"])
    cf1 = np.polyfit(xs, ys, 1); cf2 = np.polyfit(xs, ys, 2)
    p1, p2 = np.polyval(cf1, X90), np.polyval(cf2, X90)
    print(f"  {label:18s} @90M: linear {p1:+6.2f}%   quadratic {p2:+6.2f}%")
    if variant == "combo_diff":
        r = np.roots(cf2); r = [x for x in r if np.isreal(x)]
        ns = [10**np.real(x)/1e6 for x in r]
        ns = [n for n in ns if 5 < n < 5000]
        if ns: print(f"    quadratic implied diff crossover N* ~= {min(ns):,.0f}M params")
with open("../results/preregistered_90M.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["variant", "fit", "predicted_delta_pct_at_90M"])
    for variant in ["combo_conv3", "combo_diff"]:
        xs, ys = [], []
        for sz in order:
            for s in ("0", "1"):
                if (sz, variant, s) in lad:
                    xs.append(math.log10(pmd[sz] * 1e6))
                    ys.append(100.0 * (lad[(sz, variant, s)] - lad[(sz, "combo", s)]) / lad[(sz, "combo", s)])
        for i, dd in ((0, s0), (1, s1)):
            xs.append(X30); ys.append(100.0*(dd[variant]-dd["combo"])/dd["combo"])
        for deg, nm in [(1, "linear"), (2, "quadratic")]:
            w.writerow([variant, nm, f"{np.polyval(np.polyfit(xs, ys, deg), X90):.3f}"])
print("saved ../results/preregistered_90M.csv")


=== commit these numbers BEFORE any 90M run exists ===
  depthwise conv     @90M: linear -24.76%   quadratic -31.89%
  differential attn  @90M: linear  -3.90%   quadratic -38.40%
    quadratic implied diff crossover N* ~= 31M params
saved ../results/preregistered_90M.csv


## Reading this

The prediction-ablation cell is the paper's core table: *which subset of rungs and which fit shape
first calls the 29.4M outcome correctly?* The pre-registration cell commits us to ~90M numbers
before Phase 3 runs them - the difference between a retrospective curve-fit and a real prediction.
